In [ ]:
# Python import
import os
import copy
import random
import itertools
import numpy as np
import pandas as pd
import lightgbm as lgb
import warnings
import joblib
from sklearn.model_selection import train_test_split,RandomizedSearchCV
import sklearn.metrics as metrics
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.utils import resample
import lightgbm as lgb
from sklearn import svm
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from scipy.stats import uniform, randint
from ngboost import NGBClassifier
from catboost import CatBoostClassifier
from ngboost.distns import k_categorical
from xgboost import XGBClassifier

In [ ]:
def ligate_sequence(seq, add_len):
    seq = str(seq)
    seqlen = len(seq)
    if seqlen >= add_len:
        extra = seq[:add_len]
    else:
        repeat_times = add_len // seqlen
        remainder = add_len % seqlen
        extra = seq * repeat_times + seq[:remainder]
    return seq + extra

In [ ]:
def balance_dataset_by_tag(df, tag_column='tag', random_state=42):
    """
    df : pd.DataFrame
        Input dataset to process.
    tag_column : str, default 'tag'
        Column used to split the data; default is 'tag'.
    random_state : int, default 42
    Returns:
    pd.DataFrame
        Balanced dataset with equal numbers of tag 0 and tag 1 rows.
    """
    df_tag_0 = df[df[tag_column] == 0]
    df_tag_1 = df[df[tag_column] == 1]

    if len(df_tag_0) > len(df_tag_1):
        df_tag_0_downsampled = resample(df_tag_0, 
                                        replace=False,
                                        n_samples=len(df_tag_1),
                                        random_state=random_state)
        df_balanced = pd.concat([df_tag_0_downsampled, df_tag_1])
    else:
        df_tag_1_downsampled = resample(df_tag_1, 
                                        replace=False,
                                        n_samples=len(df_tag_0),
                                        random_state=random_state)
        df_balanced = pd.concat([df_tag_0, df_tag_1_downsampled])

    df_balanced = df_balanced.sample(frac=1, random_state=random_state).reset_index(drop=True)

    return df_balanced


In [ ]:
# Count the frequency of k-mer in each RNA sequence
# k-mer was normalized by total k-mer count of each RNA sequence
def _count_kmer(Dataset, k):  # k = 3, 4, 5
    
    # copy dataset
    dataset = copy.deepcopy(Dataset)
    # alphabet of nucleotide
    nucleotide = ['A', 'C', 'G', 'T']
    
    # generate k-mers
    #  k == 5:
    five = list(itertools.product(nucleotide, repeat=5))
    pentamer = [''.join(n) for n in five]
    
    #  k == 4:
    four = list(itertools.product(nucleotide, repeat=4))
    tetramer = [''.join(n) for n in four]

    # k == 3:
    three = list(itertools.product(nucleotide, repeat=3))
    threemer = [''.join(n) for n in three]
    
    # input features can be combinations of different k values
    if k == 34:
        table_kmer = dict.fromkeys(threemer, 0)
        table_kmer.update(dict.fromkeys(tetramer, 0))
    elif k == 45:
        table_kmer = dict.fromkeys(tetramer, 0)
        table_kmer.update(dict.fromkeys(pentamer, 0))
    elif k == 345:
        table_kmer = dict.fromkeys(threemer, 0)
        table_kmer.update(dict.fromkeys(tetramer, 0))
        table_kmer.update(dict.fromkeys(pentamer, 0))

    # count k-mer for each sequence
    for mer in table_kmer.keys():
        table_kmer[mer] = dataset["Sequence"].apply(lambda x: x.count(mer))
    
    # for k-mer raw count without normalization, index: nuc:1 or cyto:0
    rawcount_kmer_df = pd.DataFrame(table_kmer)
    df1_rawcount = pd.concat([rawcount_kmer_df, dataset["RNA_Symbol"]], axis=1)
    df1_rawcount.index = dataset["tag"]

    # for k-mer frequency with normalization, index: nuc:1 or cyto:0
    freq_kmer_df = rawcount_kmer_df.apply(lambda x: x / x.sum(), axis=1)
    df1 = pd.concat([freq_kmer_df, dataset["RNA_Symbol"]], axis=1)
    df1.index = dataset["tag"]

    return df1, df1_rawcount


In [ ]:
#Evaluate performance of model
def evaluate_performance(y_test, y_pred, y_prob):
    # AUROC
    auroc = metrics.roc_auc_score(y_test,y_prob)
    auroc_curve = metrics.roc_curve(y_test, y_prob)
    # AUPRC
    auprc=metrics.average_precision_score(y_test, y_prob) 
    auprc_curve=metrics.precision_recall_curve(y_test, y_prob)
    #Accuracy
    accuracy=metrics.accuracy_score(y_test,y_pred) 
    #MCC
    mcc=metrics.matthews_corrcoef(y_test,y_pred)
    
    recall=metrics.recall_score(y_test, y_pred)
    precision=metrics.precision_score(y_test, y_pred)
    f1=metrics.f1_score(y_test, y_pred)
    class_report=metrics.classification_report(y_test, y_pred,target_names = ["control","case"])

    model_perf = {"auroc":auroc,"auroc_curve":auroc_curve,
                  "auprc":auprc,"auprc_curve":auprc_curve,
                  "accuracy":accuracy, "mcc": mcc,
                  "recall":recall,"precision":precision,"f1":f1,
                  "class_report":class_report}
        
    return model_perf

In [ ]:
# Output result of evaluation
def eval_output(model_perf,path):
    with open(os.path.join(path,"Evaluate_Result_TestSet.txt"),'w') as f:
        f.write("AUROC=%s\tAUPRC=%s\tAccuracy=%s\tMCC=%s\tRecall=%s\tPrecision=%s\tf1_score=%s\n" %
               (model_perf["auroc"],model_perf["auprc"],model_perf["accuracy"],model_perf["mcc"],model_perf["recall"],model_perf["precision"],model_perf["f1"]))
        f.write("\n######NOTE#######\n")
        f.write("#According to help_documentation of sklearn.metrics.classification_report:in binary classification, recall of the positive class is also known as sensitivity; recall of the negative class is specificity#\n\n")
        f.write(model_perf["class_report"])

In [ ]:
# Plot AUROC of model
def plot_AUROC(model_perf,path):
    #get AUROC,FPR,TPR and threshold
    roc_auc = model_perf["auroc"]
    fpr,tpr,threshold = model_perf["auroc_curve"]
    #return AUROC info
    temp_df = pd.DataFrame({"FPR":fpr,"TPR":tpr})
    temp_df.to_csv(os.path.join(path,"AUROC_info.txt"),header = True,index = False, sep = '\t')
    #plot
    plt.figure()
    lw = 2
    plt.figure(figsize=(10,10))
    plt.plot(fpr, tpr, color='darkorange',
             lw=lw, label='AUROC (area = %0.2f)' % roc_auc) 
    plt.plot([0, 1], [0, 1], color='navy', lw=lw, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.0])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("AUROC of Models")
    plt.legend(loc="lower right")
    plt.savefig(os.path.join(path,"AUROC_TestSet.pdf"),format = "pdf")

In [ ]:
# Random seed
SEED = 100
random.seed(SEED)
np.random.seed(SEED)

warnings.filterwarnings(action='ignore')

# Output dir
output_dir = "../ML_models/circRNA_ML_Model_tridivided_extra4fold_Output"
if not (os.path.exists(output_dir)):
    os.mkdir(output_dir)

In [ ]:
dataset = pd.read_csv(
    '../../sample_preprocessing/circRNA/output_with_sequences.csv',
    sep='\t',
    index_col=False
)
dataset_filtered = dataset
add_len = 4 
dataset_filtered["Sequence"] = dataset_filtered["Sequence"].apply(
    lambda x: ligate_sequence(x, add_len)
)

# dataset_filtered['tag'] = dataset_filtered['Subcellular_Localization'].map({
#     "Cytosol": 0,
#     "Nucleus": 0,
#     "Extracellular vesicle": 1
# })

dataset_filtered = balance_dataset_by_tag(dataset_filtered, tag_column='tag', random_state=42)

train_df, temp_df = train_test_split(dataset_filtered, test_size=0.4, random_state=SEED, stratify=dataset_filtered['tag'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED, stratify=temp_df['tag'])

df_kmer_train, df_kmer_train_raw = _count_kmer(train_df, 345)
df_kmer_val, df_kmer_val_raw = _count_kmer(val_df, 345)
df_kmer_test, df_kmer_test_raw = _count_kmer(test_df, 345)

df_kmer_train.to_csv(os.path.join(output_dir, "train_kmer345_freq.tsv"), sep='\t')
df_kmer_train_raw.to_csv(os.path.join(output_dir, "train_kmer345_rawcount.tsv"), sep='\t')
df_kmer_val.to_csv(os.path.join(output_dir, "val_kmer345_freq.tsv"), sep='\t')
df_kmer_val_raw.to_csv(os.path.join(output_dir, "val_kmer345_rawcount.tsv"), sep='\t')
df_kmer_test.to_csv(os.path.join(output_dir, "test_kmer345_freq.tsv"), sep='\t')
df_kmer_test_raw.to_csv(os.path.join(output_dir, "test_kmer345_rawcount.tsv"), sep='\t')


In [ ]:
x_train = df_kmer_train.drop(columns=["RNA_Symbol"]).values
x_val = df_kmer_val.drop(columns=["RNA_Symbol"]).values
x_test = df_kmer_test.drop(columns=["RNA_Symbol"]).values

y_train = train_df["tag"].values
y_val = val_df["tag"].values
y_test = test_df["tag"].values

imputer = SimpleImputer(strategy='mean')
x_train = imputer.fit_transform(x_train)
x_val = imputer.transform(x_val)
x_test = imputer.transform(x_test)

In [ ]:
from sklearn.model_selection import StratifiedKFold
import numpy as np
import os

folds = []

folds.append({
    'x_train': x_train,
    'y_train': y_train,
    'x_val': x_val,
    'y_val': y_val
})

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

for train_idx, val_idx in skf.split(x_train, y_train):
    current_x_val = x_train[val_idx]
    current_y_val = y_train[val_idx]
    
    remaining_x_train = x_train[train_idx]
    remaining_y_train = y_train[train_idx]
    
    current_x_train = np.vstack((remaining_x_train, x_val))
    current_y_train = np.concatenate((remaining_y_train, y_val))
    
    folds.append({
        'x_train': current_x_train,
        'y_train': current_y_train,
        'x_val': current_x_val,
        'y_val': current_y_val
    })

folds_data_dir = os.path.join(output_dir, "Folds_Data")
if not os.path.exists(folds_data_dir):
    os.makedirs(folds_data_dir, exist_ok=True)

for i, fold_data in enumerate(folds):
    fold_file_path = os.path.join(folds_data_dir, f"Fold_{i+1}_data.npz")
    np.savez_compressed(
        fold_file_path,
        x_train=fold_data['x_train'],
        y_train=fold_data['y_train'],
        x_val=fold_data['x_val'],
        y_val=fold_data['y_val']
    )
print(f"Successfully saved the 4-fold data to: {folds_data_dir}")

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd

folds_data_dir = "../ML_models/circRNA_ML_Model_tridivided_extra4fold_Output/Folds_Data"
control_model_base_dir = "../ML_models/circRNA_ML_Model_tridivided_extra4fold_Output/RandomForest"
ablation_model_base_dir = "ligated_ablation_extra4fold_output/RandomForest"

y_all = []
p_control_all = []
p_ablation_all = []

for i in range(1, 5):
    fold_file_path = os.path.join(folds_data_dir, f"Fold_{i}_data.npz")
    fold_data = np.load(fold_file_path)
    x_val = fold_data['x_val']
    y_val = fold_data['y_val']
    
    control_model_path = os.path.join(control_model_base_dir, f"RandomForest_Fold_{i}", f"best_RandomForest_model_Fold_{i}.pkl")
    ablation_model_path = os.path.join(ablation_model_base_dir, f"RandomForest_Fold_{i}", f"best_RandomForest_model_Fold_{i}.pkl")
    
    control_model = joblib.load(control_model_path)
    ablation_model = joblib.load(ablation_model_path)
    
    p_control = control_model.predict_proba(x_val)[:, 1]
    p_ablation = ablation_model.predict_proba(x_val)[:, 1]
    
    y_all.append(y_val)
    p_control_all.append(p_control)
    p_ablation_all.append(p_ablation)

y_all = np.concatenate(y_all)
p_control_all = np.concatenate(p_control_all)
p_ablation_all = np.concatenate(p_ablation_all)

print(f"Total samples (N): {len(y_all)}")

oof_results = pd.DataFrame({
    'True_Label': y_all,
    'Prob_Control': p_control_all,
    'Prob_Ablation': p_ablation_all
})

output_csv = os.path.join(ablation_model_base_dir, "OOF_predictions_comparison.csv")
oof_results.to_csv(output_csv, index=False)
print(f"OOF probability collection completed; saved to: {output_csv}")

In [ ]:
import numpy as np
import scipy.stats as st
from sklearn.metrics import roc_auc_score
from sklearn.utils import resample

# ==========================================
# ==========================================
def compute_midrank(x):
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=np.float64)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1)
        i = j
    T2 = np.empty(N, dtype=np.float64)
    T2[J] = T + 1
    return T2

def fastDeLong(predictions_sorted_transposed, label_1_count):
    m = label_1_count
    n = predictions_sorted_transposed.shape[1] - m
    positive_examples = predictions_sorted_transposed[:, :m]
    negative_examples = predictions_sorted_transposed[:, m:]
    tx = np.empty([positive_examples.shape[0], m], dtype=np.float64)
    ty = np.empty([negative_examples.shape[0], n], dtype=np.float64)
    tz = np.empty([predictions_sorted_transposed.shape[0], m + n], dtype=np.float64)
    for r in range(predictions_sorted_transposed.shape[0]):
        tx[r, :] = compute_midrank(positive_examples[r, :])
        ty[r, :] = compute_midrank(negative_examples[r, :])
        tz[r, :] = compute_midrank(predictions_sorted_transposed[r, :])
    aucs = tz[:, :m].sum(axis=1) / m / n - float(m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx[:, :]) / n
    v10 = 1.0 - (tz[:, m:] - ty[:, :]) / m
    sx = np.cov(v01)
    sy = np.cov(v10)
    delongcov = sx / m + sy / n
    return aucs, delongcov

def calc_pvalue(aucs, sigma):
    l = np.array([[1, -1]])
    z = np.abs(np.diff(aucs)) / np.sqrt(np.dot(np.dot(l, sigma), l.T))
    return 2 * (1 - st.norm.cdf(z))

def calc_pvalue_onesided(aucs, sigma):
    """Calculate one-sided p-value (H1: AUC_A > AUC_B)."""
    l = np.array([[1, -1]])
    diff = aucs[0] - aucs[1]
    z = diff / np.sqrt(np.dot(np.dot(l, sigma), l.T))
    return 1 - st.norm.cdf(z)


# ==========================================
# ==========================================
y_true = np.array(y_all)
preds_A = np.array(p_control_all)
preds_B = np.array(p_ablation_all)

auc_control = roc_auc_score(y_true, preds_A)
auc_ablation = roc_auc_score(y_true, preds_B)
print(f"Overall OOF AUROC - Control: {auc_control:.4f}")
print(f"Overall OOF AUROC - Ablation: {auc_ablation:.4f}")
print(f"ΔAUROC (Control - Ablation): {auc_control - auc_ablation:.4f}")

order = np.argsort(y_true)[::-1]
preds_A_sorted = preds_A[order]
preds_B_sorted = preds_B[order]
preds_sorted_transposed = np.vstack((preds_A_sorted, preds_B_sorted))
label_1_count = np.sum(y_true == 1)

aucs, delongcov = fastDeLong(preds_sorted_transposed, label_1_count)
delong_pvalue = calc_pvalue(aucs, delongcov)[0, 0]
print(f"DeLong Test p-value: {delong_pvalue:.4e}")
delong_pvalue_onesided = calc_pvalue_onesided(aucs, delongcov)[0, 0]
print(f"DeLong Test ONE-SIDED p-value (Control > Ablation): {delong_pvalue_onesided:.4e}")

# ==========================================
# ==========================================
n_bootstraps = 2000
rng_seed = 42
bootstrapped_deltas = []

rng = np.random.RandomState(rng_seed)
indices = np.arange(len(y_true))

print("\nRunning Paired Bootstrap, this may take a moment...")
for i in range(n_bootstraps):
    boot_indices = resample(indices, random_state=rng)
    
    y_boot = y_true[boot_indices]
    preds_A_boot = preds_A[boot_indices]
    preds_B_boot = preds_B[boot_indices]
    
    if len(np.unique(y_boot)) < 2:
        continue
        
    auc_A_boot = roc_auc_score(y_boot, preds_A_boot)
    auc_B_boot = roc_auc_score(y_boot, preds_B_boot)
    
    bootstrapped_deltas.append(auc_A_boot - auc_B_boot)

ci_lower = np.percentile(bootstrapped_deltas, 2.5)
ci_upper = np.percentile(bootstrapped_deltas, 97.5)
delta_mean = np.mean(bootstrapped_deltas)

p_boot = np.mean(np.array(bootstrapped_deltas) <= 0)

print(f"\nBootstrap Results ({n_bootstraps} iterations):")
print(f"ΔAUROC Mean: {delta_mean:.4f}")
print(f"95% Confidence Interval: [{ci_lower:.4f}, {ci_upper:.4f}]")
if ci_lower > 0:
    print("Conclusion: The 95% CI is completely above 0. Control is stably better than Ablation.")
elif ci_upper < 0:
    print("Conclusion: The 95% CI is completely below 0. Ablation is stably better than Control.")
else:
    print("Conclusion: The 95% CI crosses 0. The difference is not strictly conclusive.")

from sklearn.metrics import average_precision_score, f1_score, accuracy_score

# ==========================================
# ==========================================
n_permutations = 10000
rng_perm = np.random.RandomState(42)
delta_obs_auroc = roc_auc_score(y_true, preds_A) - roc_auc_score(y_true, preds_B)
perm_deltas = []

print(f"Running Paired Permutation Test ({n_permutations} iterations)...")
for _ in range(n_permutations):
    swap_mask = rng_perm.rand(len(y_true)) > 0.5
    preds_A_perm = np.where(swap_mask, preds_B, preds_A)
    preds_B_perm = np.where(swap_mask, preds_A, preds_B)
    
    auc_A_perm = roc_auc_score(y_true, preds_A_perm)
    auc_B_perm = roc_auc_score(y_true, preds_B_perm)
    perm_deltas.append(auc_A_perm - auc_B_perm)

p_val_perm = np.mean(np.abs(perm_deltas) >= np.abs(delta_obs_auroc))
print(f"Paired Permutation Test p-value for AUROC: {p_val_perm:.4f}")
print("-" * 50)

p_val_perm_one_sided = np.mean(np.array(perm_deltas) >= delta_obs_auroc)
print(f"Paired Permutation Test ONE-SIDED p-value for AUROC: {p_val_perm_one_sided:.5f}")

In [ ]:
from sklearn.metrics import average_precision_score

# ==========================================
# ==========================================
auprc_control = average_precision_score(y_true, preds_A)
auprc_ablation = average_precision_score(y_true, preds_B)
delta_obs_auprc = auprc_control - auprc_ablation

print(f"Overall OOF AUPRC - Control: {auprc_control:.4f}")
print(f"Overall OOF AUPRC - Ablation: {auprc_ablation:.4f}")
print(f"ΔAUPRC (Control - Ablation): {delta_obs_auprc:.4f}")
print("-" * 50)

# ==========================================
# ==========================================
n_permutations = 10000
rng_perm = np.random.RandomState(42)
perm_deltas_auprc = []

print(f"Running Paired Permutation Test for AUPRC ({n_permutations} iterations)...")
for _ in range(n_permutations):
    swap_mask = rng_perm.rand(len(y_true)) > 0.5
    preds_A_perm = np.where(swap_mask, preds_B, preds_A)
    preds_B_perm = np.where(swap_mask, preds_A, preds_B)
    
    auprc_A_perm = average_precision_score(y_true, preds_A_perm)
    auprc_B_perm = average_precision_score(y_true, preds_B_perm)
    perm_deltas_auprc.append(auprc_A_perm - auprc_B_perm)

p_val_perm_auprc = np.mean(np.abs(perm_deltas_auprc) >= np.abs(delta_obs_auprc))
print(f"Paired Permutation Test TWO-SIDED p-value for AUPRC: {p_val_perm_auprc:.4f}")

p_val_perm_one_sided_auprc = np.mean(np.array(perm_deltas_auprc) >= delta_obs_auprc)
print(f"Paired Permutation Test ONE-SIDED p-value for AUPRC: {p_val_perm_one_sided_auprc:.5f}")
print("-" * 50)

# ==========================================
# ==========================================
n_bootstraps = 2000
rng_boot = np.random.RandomState(42)
bootstrapped_deltas_auprc = []
indices = np.arange(len(y_true))

print(f"Running Paired Bootstrap for AUPRC ({n_bootstraps} iterations)...")
for i in range(n_bootstraps):
    boot_indices = resample(indices, random_state=rng_boot)
    y_boot = y_true[boot_indices]
    
    if len(np.unique(y_boot)) < 2:
        continue
        
    preds_A_boot = preds_A[boot_indices]
    preds_B_boot = preds_B[boot_indices]
    
    auprc_A_boot = average_precision_score(y_boot, preds_A_boot)
    auprc_B_boot = average_precision_score(y_boot, preds_B_boot)
    
    bootstrapped_deltas_auprc.append(auprc_A_boot - auprc_B_boot)

ci_lower_auprc = np.percentile(bootstrapped_deltas_auprc, 2.5)
ci_upper_auprc = np.percentile(bootstrapped_deltas_auprc, 97.5)
delta_mean_auprc = np.mean(bootstrapped_deltas_auprc)

print(f"\nBootstrap Results for AUPRC:")
print(f"ΔAUPRC Mean: {delta_mean_auprc:.4f}")
print(f"95% Confidence Interval: [{ci_lower_auprc:.4f}, {ci_upper_auprc:.4f}]")
if ci_lower_auprc > 0:
    print("Conclusion: The 95% CI is completely above 0. Control is stably better than Ablation in AUPRC.")
elif ci_upper_auprc < 0:
    print("Conclusion: The 95% CI is completely below 0. Ablation is stably better than Control in AUPRC.")
else:
    print("Conclusion: The 95% CI crosses 0. The difference in AUPRC is not strictly conclusive.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, precision_recall_curve
import os

sns.set_theme(style="whitegrid", font_scale=1.1)
custom_palette = [ "#D89F7B", "#98B2CD",] # Control, Ablation

output_dir_vis = "."

# ==========================================
# ==========================================
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

fpr_A, tpr_A, _ = roc_curve(y_true, preds_A)
fpr_B, tpr_B, _ = roc_curve(y_true, preds_B)
prec_A, rec_A, _ = precision_recall_curve(y_true, preds_A)
prec_B, rec_B, _ = precision_recall_curve(y_true, preds_B)

# --- Subplot 1: ROC Curve ---
ax1 = axes[0]
ax1.plot(fpr_A, tpr_A, color=custom_palette[0], lw=2.5, 
         label=f'Control (AUC = {auc_control:.4f})')
ax1.plot(fpr_B, tpr_B, color=custom_palette[1], lw=2.5, 
         label=f'Ablation (AUC = {auc_ablation:.4f})')
ax1.plot([0, 1], [0, 1], color='gray', lw=1.5, linestyle='--')
ax1.set_xlabel('False Positive Rate', fontsize=15, fontweight='bold')
ax1.set_ylabel('True Positive Rate', fontsize=15, fontweight='bold')
ax1.set_title('Out-Of-Fold ROC Curve', fontsize=16, fontweight='bold')
p_text_roc = f"Permutation P = {p_val_perm_one_sided:.3f}\nDeLong P = 0.040"
ax1.text(0.48, 0.15, p_text_roc, fontsize=13, bbox=dict(facecolor='white', alpha=0.8, edgecolor='lightgray'))
ax1.legend(loc="lower right", fontsize=13, frameon=False)

# --- Subplot 2: PR Curve ---
ax2 = axes[1]
baseline = np.sum(y_true == 1) / len(y_true)
ax2.plot(rec_A, prec_A, color=custom_palette[0], lw=2.5, 
         label=f'Control (AUPRC = {auprc_control:.4f})')
ax2.plot(rec_B, prec_B, color=custom_palette[1], lw=2.5, 
         label=f'Ablation (AUPRC = {auprc_ablation:.4f})')
ax2.axhline(baseline, color='gray', lw=1.5, linestyle='--')
ax2.set_xlabel('Recall', fontsize=15, fontweight='bold')
ax2.set_ylabel('Precision', fontsize=15, fontweight='bold')
ax2.set_title('Out-Of-Fold PR Curve', fontsize=16, fontweight='bold')
p_text_pr = f"Permutation P = {p_val_perm_one_sided_auprc:.3f}\n"
ax2.text(0.45, 0.15, p_text_pr, fontsize=13, bbox=dict(facecolor='white', alpha=0.8, edgecolor='lightgray'))
ax2.legend(loc="lower right", fontsize=13, frameon=False)

sns.despine()
plt.tight_layout()
fig1_path = os.path.join(output_dir_vis, "OOF_ROC_PR_Curves.png")
plt.savefig(fig1_path, dpi=300, bbox_inches='tight')
plt.show()



In [ ]:
# ==========================================
# ==========================================
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

hist_color = "#98B2CD"

# --- Subplot 1: Delta AUROC ---
ax3 = axes[0]
sns.histplot(bootstrapped_deltas, bins=40, kde=True, color=hist_color, ax=ax3, edgecolor='white')
ax3.axvline(0, color='red', linestyle='-', lw=2, label='Zero Diff Line')
ax3.axvline(ci_lower, color='black', linestyle='--', lw=2, label='95% CI Lower')
ax3.axvline(ci_upper, color='black', linestyle=':', lw=2, label='95% CI Upper')
ax3.set_xlabel('$\Delta$ AUROC (Control - Ablation)', fontsize=15, fontweight='bold')
ax3.set_ylabel('Frequency', fontsize=15, fontweight='bold')
ax3.set_title('Bootstrap $\Delta$ AUROC Distribution', fontsize=16, fontweight='bold')
ax3.legend(fontsize=13, frameon=False)

# --- Subplot 2: Delta AUPRC ---
ax4 = axes[1]
sns.histplot(bootstrapped_deltas_auprc, bins=40, kde=True, color=hist_color, ax=ax4, edgecolor='white')
ax4.axvline(0, color='red', linestyle='-', lw=2, label='Zero Diff Line')
ax4.axvline(ci_lower_auprc, color='black', linestyle='--', lw=2, label='95% CI Lower')
ax4.axvline(ci_upper_auprc, color='black', linestyle=':', lw=2, label='95% CI Upper')
ax4.set_xlabel('$\Delta$ AUPRC (Control - Ablation)', fontsize=15, fontweight='bold')
ax4.set_ylabel('Frequency', fontsize=15, fontweight='bold')
ax4.set_title('Bootstrap $\Delta$ AUPRC Distribution', fontsize=16, fontweight='bold')
ax4.legend(fontsize=13, frameon=False)

sns.despine()
plt.tight_layout()
fig2_path = os.path.join(output_dir_vis, "Bootstrap_Delta_Distributions.png")
plt.savefig(fig2_path, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

base_dir_ablation = "ligated_ablation_extra4fold_output"
base_dir_control = "../ML_models/circRNA_ML_Model_tridivided_extra4fold_Output"

target_model = "RandomForest"
metrics_to_plot = ["AUROC", "AUPRC", "Accuracy", "F1_Score"]

data_list = []

path_ablation = os.path.join(base_dir_ablation, target_model, "All_Folds_Evaluate_Results.csv")
if os.path.exists(path_ablation):
    df_abl = pd.read_csv(path_ablation)
    for metric in metrics_to_plot:
        for val in df_abl[metric]:
            data_list.append({"Group": "Ligated Ablation", "Metric": metric, "Value": val})
            
path_control = os.path.join(base_dir_control, target_model, "All_Folds_Evaluate_Results.csv")
if os.path.exists(path_control):
    df_ctrl = pd.read_csv(path_control)
    for metric in metrics_to_plot:
        for val in df_ctrl[metric]:
            data_list.append({"Group": "Control", "Metric": metric, "Value": val})

df_all = pd.DataFrame(data_list)
df_all = pd.DataFrame(data_list)

plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid", font_scale=1.1)

custom_palette = ["#98B2CD", "#D89F7B"]

# Draw the box plot.
ax = sns.boxplot(
    data=df_all, 
    x="Metric", 
    y="Value", 
    hue="Group", 
    palette=custom_palette,
    width=0.6,
    fliersize=4,
    linewidth=1.5
)

sns.despine()

# plt.title('Random Forest Performance Comparison (4-Fold CV)', fontsize=16)
plt.ylabel('Score', fontsize=16, fontweight='bold')
plt.xlabel('')
plt.xticks(fontsize=15, fontweight='bold')
plt.yticks(fontsize=14)
plt.ylim(0.4, 1.0)

plt.legend(title='', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False, fontsize=14)
plt.tight_layout()

output_png = os.path.join(base_dir_ablation, "RandomForest_Metrics_Boxplot.png")
plt.savefig(output_png, dpi=300, bbox_inches='tight')
print(f"Plot saved to: {output_png}")
plt.show()